# Analisis Jaringan Kolaborasi Dosen Informatika

Notebook ini melakukan **Exploratory Network Analysis** lengkap terhadap dataset publikasi ilmiah dosen Departemen Informatika/Ilmu Komputer. Pipeline dibagi menjadi lima fase:

1. **Data Loading & Audit** — muat semua dataset, cek kondisi awal
2. **Preprocessing** — bersihkan, normalisasi, dan integrasi data
3. **Exploratory Data Analysis (EDA)** — statistik deskriptif dan tren publikasi
4. **Network Analysis** — bangun graf kolaborasi dan hitung metrik jaringan
5. **Import ke Neo4j** — muat data ke graph database untuk query lanjutan

---

## 0. Install & Import Library

Semua library yang dibutuhkan diinstall dan diimport di sini. Jalankan sekali di awal.

In [ ]:
import subprocess, sys
for pkg in ['pandas', 'numpy', 'matplotlib', 'seaborn', 'networkx', 'neo4j', 'tqdm']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('Semua library berhasil diinstall.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
import warnings
import re
from collections import Counter
from itertools import combinations

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:.4f}'.format)

plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

PALETTE = ['#2E4057', '#048A81', '#54C6EB', '#EF946C', '#C4A77D']
print('Import selesai. Siap analisis.')

---
## FASE 1 — Data Loading & Audit

Muat ketiga file CSV dan lakukan audit awal: jumlah baris, kolom, tipe data, dan missing values. Pastikan file CSV berada di folder yang sama dengan notebook ini.

In [ ]:
PATH_DOSEN   = 'dosen_infokom_final.csv'
PATH_SCHOLAR = 'dosen_papers_scholar.csv'
PATH_SCOPUS  = 'dosen_papers_scopus.csv'

dosen   = pd.read_csv(PATH_DOSEN)
scholar = pd.read_csv(PATH_SCHOLAR)
scopus  = pd.read_csv(PATH_SCOPUS)

print('=== RINGKASAN DATASET ===')
print(f'  dosen_infokom_final  : {dosen.shape[0]:>5} baris  x {dosen.shape[1]} kolom')
print(f'  dosen_papers_scholar : {scholar.shape[0]:>5} baris  x {scholar.shape[1]} kolom')
print(f'  dosen_papers_scopus  : {scopus.shape[0]:>5} baris  x {scopus.shape[1]} kolom')
print(f'  Total publikasi raw  : {scholar.shape[0] + scopus.shape[0]:>5} paper')

Cek detail tipe data setiap kolom dari masing-masing file.

In [ ]:
for label, df in [('DOSEN', dosen), ('SCHOLAR', scholar), ('SCOPUS', scopus)]:
    print(f'--- {label} ---')
    print(df.dtypes.to_string())
    print()

Audit missing values secara lengkap. Persentase missing membantu kita memutuskan apakah kolom tersebut bisa dipakai atau perlu di-drop.

In [ ]:
def audit_missing(df, label):
    total   = len(df)
    missing = df.isnull().sum()
    pct     = (missing / total * 100).round(1)
    report  = pd.DataFrame({'Missing': missing, 'Persen (%)': pct})
    report  = report[report['Missing'] > 0].sort_values('Missing', ascending=False)
    print(f'=== Missing Values: {label} ===')
    if report.empty:
        print('  Tidak ada missing values.')
    else:
        print(report.to_string())
    print()

audit_missing(dosen,   'dosen_infokom_final')
audit_missing(scholar, 'dosen_papers_scholar')
audit_missing(scopus,  'dosen_papers_scopus')

Cek duplikat — paper yang muncul lebih dari sekali dalam dataset yang sama dapat memengaruhi hasil perhitungan.

In [ ]:
dup_scholar = scholar.duplicated(subset='citation_id').sum()
dup_scopus  = scopus.duplicated(subset='DOI').sum() if 'DOI' in scopus.columns else 0
dup_dosen   = dosen.duplicated(subset='nidn').sum()

print(f'Duplikat di scholar (citation_id) : {dup_scholar}')
print(f'Duplikat di scopus  (DOI)         : {dup_scopus}')
print(f'Duplikat di dosen   (nidn)        : {dup_dosen}')

---
## FASE 2 — Preprocessing

Fase ini membersihkan dan menyiapkan semua data agar siap dianalisis. Semua langkah dikerjakan secara berurutan dalam satu alur yang konsisten.

### 2.1 Normalisasi Nama Dosen

Nama di kolom `Authors` tidak selalu persis sama dengan nama di tabel dosen karena perbedaan gelar, singkatan, atau format. Fungsi normalisasi ini menghapus gelar akademik dan menyeragamkan format title-case, lalu langsung disimpan ke kolom `nama_norm_clean` di dataframe `dosen` di cell ini sekaligus — sehingga kolom tersebut tersedia saat digunakan di cell berikutnya.

In [ ]:
def normalize_name(name):
    if pd.isna(name):
        return ''
    name = str(name).strip()
    name = re.sub(r'\s+', ' ', name)
    prefixes = r'^(Dr\.|Prof\.|Ir\.|Drs\.|Dra\.|S\.T\.|S\.Kom\.|M\.Kom\.|M\.T\.|Ph\.D\.|S\.Pd\.|M\.Pd\.|M\.Si\.|S\.Si\.|S\.Pd\.T\.|\.T\.)'
    name = re.sub(prefixes, '', name, flags=re.IGNORECASE).strip()
    name = re.sub(r',.*', '', name).strip()
    return name.title()

dosen['nama_norm_clean'] = dosen['nama_norm'].apply(normalize_name)
dosen_set    = set(dosen['nama_norm_clean'])
dosen_lookup = dict(zip(dosen['nama_norm_clean'], dosen['nama_norm']))

print(f'Total dosen dalam lookup : {len(dosen_set)}')
print()
print('Contoh hasil normalisasi:')
print(dosen[['nama_dosen', 'nama_norm', 'nama_norm_clean']].head(8).to_string(index=False))

### 2.2 Preprocessing Dataset Scholar

Filter tahun valid (2000–2026), ekstrak nama jurnal bersih dari string panjang yang mengandung volume dan nomor halaman, lalu parse kolom `Authors` menjadi list nama per paper.

In [ ]:
def extract_journal_name(j):
    if pd.isna(j):
        return 'Unknown'
    cleaned = re.sub(r'\d+.*$', '', str(j)).strip().rstrip(',')
    return cleaned if cleaned else 'Unknown'

def parse_authors_scholar(s):
    if pd.isna(s):
        return []
    return [normalize_name(a.strip()) for a in str(s).split(',') if a.strip()]

scholar_clean = scholar.copy()
scholar_clean['Year']            = pd.to_numeric(scholar_clean['Year'], errors='coerce')
scholar_clean                    = scholar_clean[scholar_clean['Year'].between(2000, 2026)].copy()
scholar_clean['Year']            = scholar_clean['Year'].astype(int)
scholar_clean['Journal_clean']   = scholar_clean['Journal'].apply(extract_journal_name)
scholar_clean['source_db']       = 'Scholar'
scholar_clean['author_list']     = scholar_clean['Authors'].apply(parse_authors_scholar)
scholar_clean['infokom_authors'] = scholar_clean['author_list'].apply(
    lambda lst: [a for a in lst if a in dosen_set]
)

scholar_with_dosen = scholar_clean[scholar_clean['infokom_authors'].map(len) > 0].copy()

print(f'Scholar raw                          : {len(scholar):,} paper')
print(f'Scholar setelah filter tahun valid   : {len(scholar_clean):,} paper')
print(f'Scholar dengan min 1 dosen Infokom   : {len(scholar_with_dosen):,} paper')
print(f'Rentang tahun                        : {scholar_clean["Year"].min()} - {scholar_clean["Year"].max()}')

### 2.3 Preprocessing Dataset Scopus

Scopus menggunakan pemisah titik-koma (`;`) untuk nama author, berbeda dari Scholar yang menggunakan koma. Proses preprocessing sama tetapi dengan parser yang sesuai format Scopus.

In [ ]:
def parse_authors_scopus(s):
    if pd.isna(s):
        return []
    return [normalize_name(a.strip()) for a in str(s).split(';') if a.strip()]

scopus_clean = scopus.copy()
scopus_clean['Year']            = pd.to_numeric(scopus_clean['Year'], errors='coerce').astype('Int64')
scopus_clean                    = scopus_clean[scopus_clean['Year'].between(2000, 2026)].copy()
scopus_clean['Journal_clean']   = scopus_clean['Journal'].fillna('Unknown')
scopus_clean['source_db']       = 'Scopus'
scopus_clean['paper_id']        = 'scopus_' + scopus_clean.index.astype(str)
scopus_clean['author_list']     = scopus_clean['Authors'].apply(parse_authors_scopus)
scopus_clean['infokom_authors'] = scopus_clean['author_list'].apply(
    lambda lst: [a for a in lst if a in dosen_set]
)

scopus_with_dosen = scopus_clean[scopus_clean['infokom_authors'].map(len) > 0].copy()

print(f'Scopus raw                           : {len(scopus):,} paper')
print(f'Scopus setelah filter tahun valid    : {len(scopus_clean):,} paper')
print(f'Scopus dengan min 1 dosen Infokom    : {len(scopus_with_dosen):,} paper')
print(f'Tipe dokumen:')
print(scopus_clean['Document Type'].value_counts().to_string())

### 2.4 Bangun Tabel Relasi Dosen–Paper (Gabungan)

Kolom `infokom_authors` berisi list nama dosen per paper. Kita explode list ini sehingga setiap baris merepresentasikan satu relasi dosen–paper. Setelah Scholar dan Scopus digabung, kita merge dengan data profil dosen untuk mendapatkan prodi dan ID akademik.

In [ ]:
scholar_exploded = (
    scholar_with_dosen
    .explode('infokom_authors')
    [['citation_id', 'Title', 'Year', 'Journal_clean', 'infokom_authors', 'source_db']]
    .rename(columns={'citation_id': 'paper_id', 'infokom_authors': 'nama_norm_clean'})
)

scopus_exploded = (
    scopus_with_dosen
    .explode('infokom_authors')
    [['paper_id', 'Title', 'Year', 'Journal_clean', 'infokom_authors', 'source_db']]
    .rename(columns={'infokom_authors': 'nama_norm_clean'})
)

relasi = pd.concat([scholar_exploded, scopus_exploded], ignore_index=True)
relasi = relasi.dropna(subset=['nama_norm_clean'])
relasi = relasi[relasi['nama_norm_clean'].str.strip() != ''].copy()

relasi = relasi.merge(
    dosen[['nama_norm_clean', 'prodi', 'scholar_id', 'scopus_id']],
    on='nama_norm_clean',
    how='left'
)

print(f'Total relasi dosen-paper : {len(relasi):,}')
print(f'Dosen unik terlibat      : {relasi["nama_norm_clean"].nunique()}')
print(f'Paper unik               : {relasi["paper_id"].nunique():,}')
print()
print(relasi.head(6).to_string(index=False))

---
## FASE 3 — Exploratory Data Analysis (EDA)

Analisis statistik deskriptif dan visualisasi tren publikasi, produktivitas dosen, distribusi per prodi, dan topik riset.

### 3.1 Distribusi Dosen per Program Studi

Berapa dosen yang dimiliki masing-masing prodi? Ini penting untuk kontekstualisasi hasil analisis jaringan.

In [ ]:
prodi_dist = dosen['prodi'].value_counts()

print('=== DISTRIBUSI DOSEN PER PRODI ===')
print(prodi_dist.to_string())
print(f'\nTotal prodi  : {dosen["prodi"].nunique()}')
print(f'Total dosen  : {len(dosen)}')

In [ ]:
top_prodi = prodi_dist.head(10)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_prodi.index[::-1], top_prodi.values[::-1],
               color=PALETTE[0], edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, top_prodi.values[::-1]):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
            str(val), va='center', fontsize=10, fontweight='bold')
ax.set_xlabel('Jumlah Dosen', fontsize=11)
ax.set_title('Top 10 Program Studi berdasarkan Jumlah Dosen', fontsize=13, fontweight='bold', pad=15)
ax.set_xlim(0, top_prodi.max() * 1.15)
plt.tight_layout()
plt.savefig('plot_01_dosen_per_prodi.png', bbox_inches='tight')
plt.show()

### 3.2 Tren Publikasi per Tahun

Apakah produktivitas riset dosen Infokom meningkat atau menurun dari tahun ke tahun? Dibedakan berdasarkan sumber (Scholar vs Scopus) untuk melihat perbedaan intensitas di tiap platform.

In [ ]:
trend = (
    relasi[relasi['Year'].between(2010, 2025)]
    .groupby(['Year', 'source_db'])
    .size()
    .unstack(fill_value=0)
)

print('=== TREN PUBLIKASI PER TAHUN (relasi dosen-paper) ===')
print(trend.to_string())
print()
for col in trend.columns:
    print(f'Total {col} (2010-2025) : {trend[col].sum():,}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_map = {'Scholar': PALETTE[0], 'Scopus': PALETTE[1]}

for col in trend.columns:
    axes[0].plot(trend.index, trend[col], marker='o', linewidth=2.5,
                 label=col, color=colors_map.get(col, PALETTE[2]))
axes[0].set_title('Tren Publikasi per Tahun per Sumber', fontweight='bold')
axes[0].set_xlabel('Tahun')
axes[0].set_ylabel('Relasi Dosen-Paper')
axes[0].legend()
axes[0].set_xticks(trend.index)
axes[0].tick_params(axis='x', rotation=45)

trend_total = trend.sum(axis=1)
axes[1].fill_between(trend_total.index, trend_total.values, alpha=0.4, color=PALETTE[2])
axes[1].plot(trend_total.index, trend_total.values, marker='o', linewidth=2.5, color=PALETTE[2])
axes[1].set_title('Total Publikasi per Tahun (Gabungan)', fontweight='bold')
axes[1].set_xlabel('Tahun')
axes[1].set_ylabel('Total')
axes[1].set_xticks(trend_total.index)
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Tren Produktivitas Riset Dosen Infokom', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_02_tren_publikasi.png', bbox_inches='tight')
plt.show()

### 3.3 Top 20 Dosen Paling Produktif

Hitung jumlah paper unik per dosen dari tabel relasi, lalu tampilkan 20 teratas beserta prodi masing-masing.

In [ ]:
produktivitas = (
    relasi.groupby('nama_norm_clean')['paper_id']
    .nunique()
    .reset_index(name='total_paper')
)
produktivitas = produktivitas.merge(
    dosen[['nama_norm_clean', 'prodi']], on='nama_norm_clean', how='left'
)
produktivitas = produktivitas.sort_values('total_paper', ascending=False).reset_index(drop=True)

print('=== TOP 20 DOSEN PALING PRODUKTIF ===')
print(produktivitas.head(20)[['nama_norm_clean', 'total_paper', 'prodi']].to_string(index=False))
print(f'\nRata-rata paper per dosen : {produktivitas["total_paper"].mean():.1f}')
print(f'Median paper per dosen    : {produktivitas["total_paper"].median():.1f}')
print(f'Maksimum (1 dosen)        : {produktivitas["total_paper"].max()}')
print(f'Minimum (1 dosen)         : {produktivitas["total_paper"].min()}')

In [ ]:
top20  = produktivitas.head(20)
colors = [PALETTE[0] if i < 3 else PALETTE[1] if i < 10 else PALETTE[2] for i in range(len(top20))]

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(top20['nama_norm_clean'][::-1], top20['total_paper'][::-1],
               color=colors[::-1], edgecolor='white')
for bar, val in zip(bars, top20['total_paper'][::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            str(val), va='center', fontsize=9, fontweight='bold')
legend_patches = [
    mpatches.Patch(color=PALETTE[0], label='Rank 1-3'),
    mpatches.Patch(color=PALETTE[1], label='Rank 4-10'),
    mpatches.Patch(color=PALETTE[2], label='Rank 11-20'),
]
ax.legend(handles=legend_patches, loc='lower right')
ax.set_xlabel('Jumlah Paper Unik', fontsize=11)
ax.set_title('Top 20 Dosen Infokom Paling Produktif', fontsize=13, fontweight='bold', pad=15)
ax.set_xlim(0, top20['total_paper'].max() * 1.12)
plt.tight_layout()
plt.savefig('plot_03_top20_produktif.png', bbox_inches='tight')
plt.show()

### 3.4 Distribusi Produktivitas & Fenomena Pareto

Distribusi jumlah paper per dosen biasanya tidak merata. Fenomena Pareto (80/20) sangat umum di akademik: sebagian kecil dosen menghasilkan sebagian besar publikasi.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(produktivitas['total_paper'], bins=20, color=PALETTE[0], edgecolor='white', linewidth=0.7)
axes[0].axvline(produktivitas['total_paper'].mean(), color=PALETTE[3], linestyle='--',
                linewidth=2, label=f'Rata-rata: {produktivitas["total_paper"].mean():.1f}')
axes[0].axvline(produktivitas['total_paper'].median(), color=PALETTE[4], linestyle='--',
                linewidth=2, label=f'Median: {produktivitas["total_paper"].median():.1f}')
axes[0].set_xlabel('Jumlah Paper per Dosen')
axes[0].set_ylabel('Frekuensi')
axes[0].set_title('Distribusi Produktivitas Dosen', fontweight='bold')
axes[0].legend()

by_source = relasi.drop_duplicates(subset=['paper_id', 'source_db']).groupby('source_db').size()
axes[1].pie(by_source.values, labels=by_source.index, autopct='%1.1f%%',
            colors=[PALETTE[0], PALETTE[1]], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporsi Paper Unik: Scholar vs Scopus', fontweight='bold')

plt.tight_layout()
plt.savefig('plot_04_distribusi_produktivitas.png', bbox_inches='tight')
plt.show()

sorted_prod = produktivitas.sort_values('total_paper', ascending=False)
cumsum  = sorted_prod['total_paper'].cumsum() / sorted_prod['total_paper'].sum()
pct_80  = (cumsum <= 0.80).sum() + 1
print(f'Hukum Pareto: {pct_80} dosen ({pct_80/len(produktivitas)*100:.1f}%) menghasilkan 80% dari total publikasi.')

### 3.5 Top 15 Jurnal dan Venue Publikasi

Jurnal atau konferensi apa yang paling sering dipilih dosen Infokom? Menunjukkan orientasi riset: nasional atau internasional.

In [ ]:
top_jurnal = (
    relasi.drop_duplicates(subset='paper_id')
    .groupby(['Journal_clean', 'source_db'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(15)
)

print('=== TOP 15 JURNAL/VENUE PUBLIKASI ===')
print(top_jurnal.to_string(index=False))

In [ ]:
top_jurnal['Journal_short'] = top_jurnal['Journal_clean'].str[:52]
color_map  = {'Scholar': PALETTE[0], 'Scopus': PALETTE[1]}
bar_colors = [color_map.get(src, PALETTE[2]) for src in top_jurnal['source_db']]

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(top_jurnal['Journal_short'][::-1], top_jurnal['count'][::-1],
               color=bar_colors[::-1], edgecolor='white')
for bar, val in zip(bars, top_jurnal['count'][::-1]):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
            str(val), va='center', fontsize=9)
legend_patches = [mpatches.Patch(color=PALETTE[0], label='Scholar'),
                  mpatches.Patch(color=PALETTE[1], label='Scopus')]
ax.legend(handles=legend_patches)
ax.set_xlabel('Jumlah Paper')
ax.set_title('Top 15 Jurnal/Venue Publikasi Dosen Infokom', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('plot_05_top_jurnal.png', bbox_inches='tight')
plt.show()

### 3.6 Analisis Keyword / Topik Riset (Scopus)

Scopus memiliki data Keywords yang terstruktur. Analisis ini menunjukkan topik riset apa yang paling dominan berdasarkan frekuensi kemunculan keyword di seluruh paper Scopus.

In [ ]:
all_keywords = []
for kw_str in scopus_clean['Keywords'].dropna():
    kws = [k.strip().lower() for k in str(kw_str).split(';') if k.strip()]
    all_keywords.extend(kws)

kw_counter = Counter(all_keywords)
top_kw     = pd.DataFrame(kw_counter.most_common(25), columns=['keyword', 'count'])

print('=== TOP 25 KEYWORD RISET (SCOPUS) ===')
print(top_kw.to_string(index=False))

In [ ]:
top20_kw  = top_kw.head(20)
cmap_vals = plt.cm.Blues(np.linspace(0.4, 0.9, len(top20_kw)))

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top20_kw['keyword'][::-1], top20_kw['count'][::-1],
               color=cmap_vals, edgecolor='white')
for bar, val in zip(bars, top20_kw['count'][::-1]):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
            str(val), va='center', fontsize=9)
ax.set_xlabel('Frekuensi')
ax.set_title('Top 20 Keyword Riset Dosen Infokom (Scopus)', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('plot_06_top_keywords.png', bbox_inches='tight')
plt.show()

---
## FASE 4 — Network Analysis

Bangun **graf kolaborasi** di mana setiap node adalah dosen dan setiap edge menyatakan mereka pernah menulis paper bersama. Bobot edge mencerminkan frekuensi kolaborasi.

### 4.1 Bangun Graf Kolaborasi

Untuk setiap paper yang ditulis oleh lebih dari satu dosen Infokom, kita buat edge antara semua pasangan penulisnya menggunakan kombinasi 2 dari N penulis.

In [ ]:
paper_authors = (
    relasi.groupby('paper_id')['nama_norm_clean']
    .apply(list)
    .reset_index()
)
paper_authors = paper_authors[paper_authors['nama_norm_clean'].map(len) >= 2]

edge_counter = Counter()
for _, row in paper_authors.iterrows():
    authors = sorted(set(row['nama_norm_clean']))
    for pair in combinations(authors, 2):
        edge_counter[pair] += 1

G = nx.Graph()
for dosen_name in relasi['nama_norm_clean'].unique():
    prodi_val = dosen[dosen['nama_norm_clean'] == dosen_name]['prodi'].values
    prodi_val = prodi_val[0] if len(prodi_val) > 0 else 'Unknown'
    G.add_node(dosen_name, prodi=prodi_val)

for (u, v), w in edge_counter.items():
    G.add_edge(u, v, weight=w)

components   = list(nx.connected_components(G))
largest_comp = max(components, key=len)

print('=== STATISTIK GRAF KOLABORASI ===')
print(f'Jumlah node (dosen)              : {G.number_of_nodes()}')
print(f'Jumlah edge (pasangan kolaborasi): {G.number_of_edges()}')
print(f'Density graf                     : {nx.density(G):.4f}')
print(f'Apakah connected                 : {nx.is_connected(G)}')
print(f'Jumlah komponen terpisah         : {len(components)}')
print(f'Komponen terbesar                : {len(largest_comp)} dosen')
print(f'Rata-rata bobot edge             : {np.mean([d["weight"] for _,_,d in G.edges(data=True)]):.2f}')

### 4.2 Hitung Metrik Sentralitas

Empat metrik dihitung pada komponen terbesar (Gcc) karena betweenness dan closeness membutuhkan graph yang connected:

- **Degree Centrality**: seberapa banyak koneksi langsung
- **Betweenness Centrality**: seberapa sering menjadi jembatan antar dosen lain
- **Closeness Centrality**: seberapa dekat ke semua node lain
- **Clustering Coefficient**: seberapa erat kelompok kolaborasi di sekitarnya

In [ ]:
Gcc = G.subgraph(largest_comp).copy()

degree_cent  = nx.degree_centrality(Gcc)
betweenness  = nx.betweenness_centrality(Gcc, weight='weight', normalized=True)
closeness    = nx.closeness_centrality(Gcc)
clustering   = nx.clustering(Gcc, weight='weight')
degree_raw   = dict(Gcc.degree(weight='weight'))

metrics_df = pd.DataFrame({
    'dosen'             : list(Gcc.nodes()),
    'weighted_degree'   : [degree_raw[n]  for n in Gcc.nodes()],
    'degree_centrality' : [degree_cent[n] for n in Gcc.nodes()],
    'betweenness'       : [betweenness[n] for n in Gcc.nodes()],
    'closeness'         : [closeness[n]   for n in Gcc.nodes()],
    'clustering'        : [clustering[n]  for n in Gcc.nodes()],
})

metrics_df = metrics_df.merge(
    dosen[['nama_norm_clean', 'prodi']],
    left_on='dosen', right_on='nama_norm_clean', how='left'
).drop(columns='nama_norm_clean')

metrics_df = metrics_df.sort_values('betweenness', ascending=False).reset_index(drop=True)

print('=== TOP 15 DOSEN — BETWEENNESS CENTRALITY ===')
print('(Nilai tinggi = berperan sebagai jembatan kolaborasi antar kelompok)')
print()
print(metrics_df.head(15)[['dosen', 'weighted_degree', 'betweenness', 'closeness', 'clustering']]
      .to_string(index=False))

Statistik deskriptif semua metrik sentralitas.

In [ ]:
print('=== STATISTIK DESKRIPTIF METRIK JARINGAN ===')
print(metrics_df[['weighted_degree', 'degree_centrality', 'betweenness', 'closeness', 'clustering']]
      .describe().to_string())

### 4.3 Visualisasi Sentralitas

Scatter plot Degree vs Betweenness membantu mengidentifikasi empat tipe dosen dalam jaringan: banyak kolaborasi dan jadi jembatan, banyak kolaborasi tapi di klaster tertutup, sedikit tapi strategis, atau perifer.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sc = axes[0].scatter(
    metrics_df['degree_centrality'],
    metrics_df['betweenness'],
    c=metrics_df['clustering'],
    cmap='coolwarm',
    s=metrics_df['weighted_degree'] * 3 + 30,
    alpha=0.75,
    edgecolors='white',
    linewidths=0.5
)
plt.colorbar(sc, ax=axes[0], label='Clustering Coefficient')
top5 = metrics_df.nlargest(5, 'betweenness')
for _, row in top5.iterrows():
    axes[0].annotate(row['dosen'], (row['degree_centrality'], row['betweenness']),
                     fontsize=7, xytext=(5, 3), textcoords='offset points')
axes[0].set_xlabel('Degree Centrality')
axes[0].set_ylabel('Betweenness Centrality')
axes[0].set_title('Degree vs Betweenness Centrality\n(ukuran titik = weighted degree)', fontweight='bold')

top15d = metrics_df.nlargest(15, 'weighted_degree')
axes[1].barh(top15d['dosen'][::-1], top15d['weighted_degree'][::-1],
             color=PALETTE[1], edgecolor='white')
axes[1].set_xlabel('Weighted Degree (total bobot kolaborasi)')
axes[1].set_title('Top 15 Dosen — Weighted Degree', fontweight='bold')

plt.suptitle('Analisis Sentralitas Jaringan Kolaborasi', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plot_07_sentralitas.png', bbox_inches='tight')
plt.show()

### 4.4 Deteksi Komunitas Kolaborasi

Algoritma Greedy Modularity mengelompokkan dosen-dosen yang cenderung berkolaborasi satu sama lain ke dalam komunitas. Setiap komunitas kemungkinan merepresentasikan satu klaster riset atau bidang keahlian.

In [ ]:
communities        = list(nx.community.greedy_modularity_communities(Gcc, weight='weight'))
communities_sorted = sorted(communities, key=len, reverse=True)

node_community = {}
for i, comm in enumerate(communities_sorted):
    for node in comm:
        node_community[node] = i

nx.set_node_attributes(Gcc, node_community, 'community')

print(f'Jumlah komunitas terdeteksi : {len(communities_sorted)}')
print()
for i, comm in enumerate(communities_sorted[:8]):
    members = sorted(list(comm))
    preview = members[:5]
    sisa    = f' ... (+{len(members)-5} lagi)' if len(members) > 5 else ''
    print(f'Komunitas {i+1:2} ({len(comm):3} dosen): {", ".join(preview)}{sisa}')

### 4.5 Visualisasi Graf Kolaborasi

Graf divisualisasikan dengan warna node berdasarkan komunitas, ukuran node berdasarkan betweenness centrality, dan ketebalan edge berdasarkan jumlah kolaborasi. Hanya edge dengan bobot >= 2 yang ditampilkan agar visualisasi tidak terlalu padat.

In [ ]:
min_collab = 2
G_filtered = nx.Graph()
for n, data in Gcc.nodes(data=True):
    G_filtered.add_node(n, **data)
for u, v, d in Gcc.edges(data=True):
    if d.get('weight', 1) >= min_collab:
        G_filtered.add_edge(u, v, **d)
G_filtered.remove_nodes_from(list(nx.isolates(G_filtered)))

print(f'Graf setelah filter (min {min_collab} kolaborasi):')
print(f'  Node : {G_filtered.number_of_nodes()}')
print(f'  Edge : {G_filtered.number_of_edges()}')

In [ ]:
pos              = nx.spring_layout(G_filtered, seed=42, k=2.5)
community_colors = plt.cm.tab20.colors
node_colors      = [community_colors[node_community.get(n, 0) % len(community_colors)]
                    for n in G_filtered.nodes()]
node_sizes       = [betweenness.get(n, 0.001) * 8000 + 100 for n in G_filtered.nodes()]
edge_weights     = [G_filtered[u][v].get('weight', 1) for u, v in G_filtered.edges()]
edge_widths      = [0.5 + w * 0.4 for w in edge_weights]

fig, ax = plt.subplots(figsize=(16, 12))
ax.set_facecolor('#F8F9FA')

nx.draw_networkx_edges(G_filtered, pos, ax=ax, width=edge_widths,
                       edge_color='#AAAAAA', alpha=0.5)
nx.draw_networkx_nodes(G_filtered, pos, ax=ax, node_color=node_colors,
                       node_size=node_sizes, alpha=0.9,
                       linewidths=1, edgecolors='white')

top_nodes    = sorted(betweenness.keys(), key=lambda n: betweenness[n], reverse=True)[:15]
top_in_graph = [n for n in top_nodes if n in G_filtered.nodes()]
nx.draw_networkx_labels(G_filtered, pos, labels={n: n for n in top_in_graph},
                        ax=ax, font_size=7, font_weight='bold', font_color='#1a1a1a')

num_comm = len(set(node_community.get(n, 0) for n in G_filtered.nodes()))
patches  = [mpatches.Patch(color=community_colors[i % len(community_colors)],
                            label=f'Komunitas {i+1}')
            for i in range(min(num_comm, 8))]
ax.legend(handles=patches, loc='lower left', fontsize=8,
          title='Kelompok Kolaborasi', title_fontsize=9)
ax.set_title(
    'Graf Kolaborasi Dosen Infokom\n'
    '(Ukuran node = Betweenness | Warna = Komunitas | Tebal edge = Frekuensi kolaborasi)',
    fontsize=13, fontweight='bold', pad=20
)
ax.axis('off')
plt.tight_layout()
plt.savefig('plot_08_graf_kolaborasi.png', bbox_inches='tight', dpi=150)
plt.show()

### 4.6 Kolaborasi Lintas Prodi

Seberapa banyak paper yang melibatkan dosen dari prodi berbeda? Dan pasangan prodi mana yang paling sering berkolaborasi?

In [ ]:
paper_prodi     = relasi.groupby('paper_id')['prodi'].apply(list)
cross_prodi_ctr = Counter()

for _, prodi_list in paper_prodi.items():
    prodi_unique = sorted(set(p for p in prodi_list if pd.notna(p)))
    for pair in combinations(prodi_unique, 2):
        cross_prodi_ctr[pair] += 1

cross_df = pd.DataFrame([
    {'prodi_a': k[0], 'prodi_b': k[1], 'kolaborasi': v}
    for k, v in cross_prodi_ctr.most_common(15)
])

cross_single = relasi.groupby('paper_id')['prodi'].nunique()
lintas  = (cross_single > 1).sum()
satu    = (cross_single == 1).sum()
total_p = len(cross_single)

print(f'Paper dengan kolaborasi LINTAS prodi  : {lintas:,} ({lintas/total_p*100:.1f}%)')
print(f'Paper dari satu prodi saja            : {satu:,} ({satu/total_p*100:.1f}%)')
print()
print('=== TOP 15 PASANGAN PRODI PALING BANYAK BERKOLABORASI ===')
print(cross_df.to_string(index=False))

---
## FASE 5 — Import ke Neo4j

Data dimuat ke Neo4j sebagai Knowledge Graph dengan struktur:
- **Node**: `(:Dosen)`, `(:Paper)`, `(:Jurnal)`, `(:Prodi)`
- **Relationship**: `MENULIS`, `DITERBITKAN_DI`, `MENGAJAR_DI`, `BERKOLABORASI`

Ganti `NEO4J_PASSWORD` dengan password yang kamu buat saat membuat instance DosenInfokom di Neo4j Desktop.

In [ ]:
from neo4j import GraphDatabase

NEO4J_URI      = 'neo4j://127.0.0.1:7687'
NEO4J_USER     = 'neo4j'
NEO4J_PASSWORD = 'ISI_PASSWORD_KAMU_DISINI'

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

try:
    driver.verify_connectivity()
    print('Koneksi ke Neo4j berhasil!')
    print(f'URI: {NEO4J_URI}')
except Exception as e:
    print(f'Koneksi GAGAL: {e}')

### 5.1 Reset Database & Buat Constraint

Hapus semua data lama, lalu buat unique constraint agar MERGE tidak menciptakan node duplikat.

In [ ]:
with driver.session() as session:
    session.run('MATCH (n) DETACH DELETE n')
    print('Database dikosongkan.')
    for c in [
        'CREATE CONSTRAINT IF NOT EXISTS FOR (d:Dosen)  REQUIRE d.nama     IS UNIQUE',
        'CREATE CONSTRAINT IF NOT EXISTS FOR (p:Paper)  REQUIRE p.paper_id IS UNIQUE',
        'CREATE CONSTRAINT IF NOT EXISTS FOR (j:Jurnal) REQUIRE j.nama     IS UNIQUE',
        'CREATE CONSTRAINT IF NOT EXISTS FOR (pr:Prodi) REQUIRE pr.nama    IS UNIQUE',
    ]:
        session.run(c)
    print('Constraint berhasil dibuat.')

### 5.2 Fungsi Batch Import

Neo4j bekerja optimal dengan batch insert. Fungsi ini memecah data menjadi potongan kecil untuk menghindari timeout dan memori berlebih.

In [ ]:
def batch_run(session, query, data, batch_size=200):
    total = 0
    for i in range(0, len(data), batch_size):
        session.run(query, rows=data[i:i+batch_size])
        total += len(data[i:i+batch_size])
    return total

print('Fungsi batch_run siap.')

### 5.3 Import Dosen, Paper, & Semua Relasi

In [ ]:
dosen_records = [
    {
        'nama'        : row['nama_norm_clean'],
        'nama_lengkap': row['nama_dosen'],
        'prodi'       : row['prodi'],
        'scholar_id'  : str(row['scholar_id']) if pd.notna(row['scholar_id']) else '',
        'scopus_id'   : str(row['scopus_id'])  if pd.notna(row['scopus_id'])  else '',
        'nidn'        : str(row['nidn'])        if pd.notna(row['nidn'])        else '',
    }
    for _, row in dosen.iterrows()
]

with driver.session() as session:
    n = batch_run(session, '''
        UNWIND $rows AS row
        MERGE (d:Dosen {nama: row.nama})
          SET d.nama_lengkap = row.nama_lengkap,
              d.scholar_id   = row.scholar_id,
              d.scopus_id    = row.scopus_id,
              d.nidn         = row.nidn
        MERGE (pr:Prodi {nama: row.prodi})
        MERGE (d)-[:MENGAJAR_DI]->(pr)
    ''', dosen_records)
    print(f'Berhasil import {n} dosen beserta relasi ke Prodi.')

In [ ]:
paper_meta    = relasi.drop_duplicates(subset='paper_id')[['paper_id', 'Title', 'Year', 'Journal_clean', 'source_db']]
paper_records = [
    {
        'paper_id': str(row['paper_id']),
        'title'   : str(row['Title'])[:300]        if pd.notna(row['Title'])         else '',
        'year'    : int(row['Year'])                if pd.notna(row['Year'])          else 0,
        'journal' : str(row['Journal_clean'])       if pd.notna(row['Journal_clean']) else 'Unknown',
        'source'  : str(row['source_db']),
    }
    for _, row in paper_meta.iterrows()
]

with driver.session() as session:
    n = batch_run(session, '''
        UNWIND $rows AS row
        MERGE (p:Paper {paper_id: row.paper_id})
          SET p.title  = row.title,
              p.year   = row.year,
              p.source = row.source
        MERGE (j:Jurnal {nama: row.journal})
        MERGE (p)-[:DITERBITKAN_DI]->(j)
    ''', paper_records)
    print(f'Berhasil import {n} paper beserta relasi ke Jurnal.')

In [ ]:
relasi_records = [
    {'dosen': row['nama_norm_clean'], 'paper_id': str(row['paper_id'])}
    for _, row in relasi.iterrows()
    if pd.notna(row['nama_norm_clean']) and str(row['nama_norm_clean']).strip() != ''
]

with driver.session() as session:
    n = batch_run(session, '''
        UNWIND $rows AS row
        MATCH (d:Dosen {nama: row.dosen})
        MATCH (p:Paper {paper_id: row.paper_id})
        MERGE (d)-[:MENULIS]->(p)
    ''', relasi_records)
    print(f'Berhasil import {n} relasi MENULIS.')

In [ ]:
collab_records = [
    {'dosen_a': pair[0], 'dosen_b': pair[1], 'weight': count}
    for pair, count in edge_counter.items()
]

with driver.session() as session:
    n = batch_run(session, '''
        UNWIND $rows AS row
        MATCH (a:Dosen {nama: row.dosen_a})
        MATCH (b:Dosen {nama: row.dosen_b})
        MERGE (a)-[r:BERKOLABORASI]-(b)
          SET r.weight = row.weight
    ''', collab_records)
    print(f'Berhasil import {n} relasi BERKOLABORASI.')

### 5.4 Verifikasi & Query Sampel dari Neo4j

Jalankan beberapa query Cypher untuk memastikan semua data masuk dengan benar.

In [ ]:
checks = {
    'Jumlah Dosen'      : 'MATCH (d:Dosen)  RETURN count(d) AS total',
    'Jumlah Paper'      : 'MATCH (p:Paper)  RETURN count(p) AS total',
    'Jumlah Jurnal'     : 'MATCH (j:Jurnal) RETURN count(j) AS total',
    'Jumlah Prodi'      : 'MATCH (pr:Prodi) RETURN count(pr) AS total',
    'Relasi MENULIS'    : 'MATCH ()-[r:MENULIS]->()       RETURN count(r)   AS total',
    'Relasi KOLABORASI' : 'MATCH ()-[r:BERKOLABORASI]-()  RETURN count(r)/2 AS total',
}

print('=== VERIFIKASI ISI GRAPH DATABASE ===')
with driver.session() as session:
    for label, q in checks.items():
        result = session.run(q).single()
        print(f'  {label:<22} : {result["total"]}')

In [ ]:
print('=== TOP 10 DOSEN PALING PRODUKTIF (dari Neo4j) ===')
with driver.session() as session:
    for i, row in enumerate(session.run('''
        MATCH (d:Dosen)-[:MENULIS]->(p:Paper)
        RETURN d.nama AS dosen, count(p) AS total_paper
        ORDER BY total_paper DESC LIMIT 10
    '''), 1):
        print(f'  {i:2}. {row["dosen"]:<30} : {row["total_paper"]} paper')

In [ ]:
print('=== TOP 10 DOSEN PALING BANYAK BERKOLABORASI (dari Neo4j) ===')
with driver.session() as session:
    for i, row in enumerate(session.run('''
        MATCH (d:Dosen)-[r:BERKOLABORASI]-(other:Dosen)
        RETURN d.nama AS dosen,
               count(DISTINCT other) AS mitra_unik,
               sum(r.weight)         AS total_kolaborasi
        ORDER BY total_kolaborasi DESC LIMIT 10
    '''), 1):
        print(f'  {i:2}. {row["dosen"]:<30} | Mitra: {row["mitra_unik"]:3} | Total kolaborasi: {row["total_kolaborasi"]}')

In [ ]:
print('=== TOP 5 PAPER DENGAN DOSEN INFOKOM TERBANYAK (dari Neo4j) ===')
with driver.session() as session:
    for i, row in enumerate(session.run('''
        MATCH (d:Dosen)-[:MENULIS]->(p:Paper)
        WITH p, count(d) AS jumlah_dosen
        WHERE jumlah_dosen > 1
        RETURN p.title AS judul, p.year AS tahun, jumlah_dosen
        ORDER BY jumlah_dosen DESC LIMIT 5
    '''), 1):
        print(f'  {i}. [{row["tahun"]}] {str(row["judul"])[:70]}')
        print(f'     Dosen Infokom terlibat: {row["jumlah_dosen"]}')

driver.close()
print()
print('Koneksi Neo4j ditutup.')

In [ ]:
print('=== SELESAI ===')
print('Semua fase berhasil dijalankan:')
print('  [1] Data Loading & Audit        — 3 dataset dimuat dan diaudit')
print('  [2] Preprocessing & Normalisasi — nama dosen dinormalisasi, relasi dibangun')
print('  [3] Exploratory Data Analysis   — 6 visualisasi dihasilkan')
print('  [4] Network Analysis            — graf, sentralitas, komunitas')
print('  [5] Import ke Neo4j             — 4 node type dan 4 relasi type dimuat')

---
## Query Cypher Lanjutan di Neo4j Browser

Setelah notebook selesai, buka Neo4j Desktop -> Query, dan coba query-query ini:

```cypher
// Mitra kolaborasi terbanyak satu dosen tertentu
MATCH (d:Dosen {nama: 'Yuni Yamasari'})-[r:BERKOLABORASI]-(other)
RETURN other.nama, r.weight ORDER BY r.weight DESC LIMIT 10

// Paper yang ditulis dosen dari dua prodi berbeda
MATCH (d1:Dosen)-[:MENGAJAR_DI]->(pr1:Prodi),
      (d2:Dosen)-[:MENGAJAR_DI]->(pr2:Prodi),
      (d1)-[:MENULIS]->(p:Paper)<-[:MENULIS]-(d2)
WHERE pr1 <> pr2
RETURN p.title, pr1.nama, pr2.nama LIMIT 10

// Jumlah paper per tahun satu dosen tertentu
MATCH (d:Dosen {nama: 'Aditya Prapanca'})-[:MENULIS]->(p:Paper)
RETURN p.year AS tahun, count(p) AS total ORDER BY tahun

// Dosen yang tidak pernah berkolaborasi dengan siapapun
MATCH (d:Dosen) WHERE NOT (d)-[:BERKOLABORASI]-() RETURN d.nama
```